# PC.3 - Tradução de emojis em conversa de WhatsApp com NLTK

In [1]:
from pathlib import Path
from collections import Counter
import re
import unicodedata

from nltk.tokenize import TweetTokenizer

BASE_DIR = Path.cwd()
if (BASE_DIR / "preprocessing.pdf").exists():
    ATIVIDADE_DIR = BASE_DIR
elif (BASE_DIR / "atv3" / "preprocessing.pdf").exists():
    ATIVIDADE_DIR = BASE_DIR / "atv3"
else:
    raise FileNotFoundError("Não encontrei preprocessing.pdf. Execute o notebook na pasta do projeto ou em atv3.")

ARQUIVO_CONVERSA = ATIVIDADE_DIR / "conversa_whatsapp_utf8.txt"
ARQUIVO_TRADUZIDO = ATIVIDADE_DIR / "conversa_whatsapp_traduzida.txt"
TOKENIZADOR = TweetTokenizer()

ATIVIDADE_DIR

PosixPath('/home/davi-maciel/dev/Faculdade/pln/atv3')

## 1. Criação da conversa em UTF-8

O arquivo abaixo simula uma exportação textual de conversa de WhatsApp.

In [2]:
conversa_whatsapp = """[09/05/2026, 19:10] Ana: Oi pessoal! Quem topa estudar PLN hoje? 📚🤓
[09/05/2026, 19:11] Bruno: Eu topo! Mas preciso de café primeiro ☕😂
[09/05/2026, 19:12] Carla: Fechado 👍 Levo os resumos e meu 💻
[09/05/2026, 19:13] Diego: Perfeito! Depois a gente pede 🍕😋
[09/05/2026, 19:14] Ana: Combinado ❤️ Vou mandar o PDF no grupo agora.
[09/05/2026, 19:15] Bruno: Valeu! Se der erro no código eu 😭
[09/05/2026, 19:16] Carla: Calma, a gente resolve 💪✨
[09/05/2026, 19:17] Diego: Bora! 🚀
"""

ARQUIVO_CONVERSA.write_text(conversa_whatsapp, encoding="utf-8")
print(f"Arquivo criado: {ARQUIVO_CONVERSA}")
print(ARQUIVO_CONVERSA.read_text(encoding="utf-8"))

Arquivo criado: /home/davi-maciel/dev/Faculdade/pln/atv3/conversa_whatsapp_utf8.txt
[09/05/2026, 19:10] Ana: Oi pessoal! Quem topa estudar PLN hoje? 📚🤓
[09/05/2026, 19:11] Bruno: Eu topo! Mas preciso de café primeiro ☕😂
[09/05/2026, 19:12] Carla: Fechado 👍 Levo os resumos e meu 💻
[09/05/2026, 19:13] Diego: Perfeito! Depois a gente pede 🍕😋
[09/05/2026, 19:14] Ana: Combinado ❤️ Vou mandar o PDF no grupo agora.
[09/05/2026, 19:15] Bruno: Valeu! Se der erro no código eu 😭
[09/05/2026, 19:16] Carla: Calma, a gente resolve 💪✨
[09/05/2026, 19:17] Diego: Bora! 🚀



## 2. Tabela de tradução dos emojis

In [3]:
EMOJI_MAP = {
    "😂": "risada",
    "😊": "sorriso",
    "😍": "apaixonado",
    "😢": "tristeza",
    "😭": "choro",
    "😡": "raiva",
    "👍": "joinha",
    "👏": "palmas",
    "🙏": "agradecimento",
    "❤️": "amor",
    "❤": "amor",
    "🔥": "muito bom",
    "🚀": "empolgação",
    "📚": "livros/estudo",
    "🤓": "estudioso",
    "☕": "café",
    "💻": "computador",
    "🍕": "pizza",
    "😋": "saboroso",
    "💪": "força",
    "✨": "brilho",
}

for emoji, traducao in EMOJI_MAP.items():
    print(f"{emoji} -> [{traducao}]")

😂 -> [risada]
😊 -> [sorriso]
😍 -> [apaixonado]
😢 -> [tristeza]
😭 -> [choro]
😡 -> [raiva]
👍 -> [joinha]
👏 -> [palmas]
🙏 -> [agradecimento]
❤️ -> [amor]
❤ -> [amor]
🔥 -> [muito bom]
🚀 -> [empolgação]
📚 -> [livros/estudo]
🤓 -> [estudioso]
☕ -> [café]
💻 -> [computador]
🍕 -> [pizza]
😋 -> [saboroso]
💪 -> [força]
✨ -> [brilho]


## 3. Funções do programa com NLTK

In [4]:
VARIATION_SELECTORS = {"\ufe0e", "\ufe0f"}
SKIN_TONE_RANGE = range(0x1F3FB, 0x1F400)

# Tokens que não devem receber espaço antes na reconstrução da frase.
SEM_ESPACO_ANTES = {".", ",", "!", "?", ";", ":", "%", ")", "]", "}", "/"}

# Tokens que não devem receber espaço depois na reconstrução da frase.
SEM_ESPACO_DEPOIS = {"(", "[", "{", "/"}


def eh_tom_de_pele(caractere: str) -> bool:
    return ord(caractere) in SKIN_TONE_RANGE


def eh_seletor_variacao_solto(token: str) -> bool:
    """Identifica tokens formados apenas por seletores de variação Unicode."""
    return bool(token) and all(caractere in VARIATION_SELECTORS for caractere in token)


def normalizar_token_emoji(token: str) -> str:
    """Remove modificadores simples para buscar uma forma base no dicionário de emojis."""
    return "".join(
        caractere
        for caractere in token
        if caractere not in VARIATION_SELECTORS and not eh_tom_de_pele(caractere)
    )


def parece_emoji_token(token: str) -> bool:
    """Heurística para reconhecer tokens que contêm ao menos um caractere de emoji."""
    for caractere in token:
        codigo = ord(caractere)
        if 0x1F000 <= codigo <= 0x1FAFF or 0x2600 <= codigo <= 0x27BF:
            return True
    return False


def traduzir_token(token: str, mapa: dict[str, str], marcar_desconhecidos: bool = False) -> str | None:
    """Traduz um token quando ele é emoji; retorna None para seletores de variação soltos."""
    if eh_seletor_variacao_solto(token):
        return None

    if token in mapa:
        return f"[{mapa[token]}]"

    token_normalizado = normalizar_token_emoji(token)
    if token_normalizado in mapa:
        return f"[{mapa[token_normalizado]}]"

    if marcar_desconhecidos and parece_emoji_token(token):
        return f"[emoji desconhecido: {token}]"

    return token


def traduzir_tokens(tokens: list[str], mapa: dict[str, str], marcar_desconhecidos: bool = False) -> list[str]:
    """Aplica a tradução token a token e remove seletores de variação soltos."""
    tokens_traduzidos = []
    for token in tokens:
        token_traduzido = traduzir_token(token, mapa, marcar_desconhecidos)
        if token_traduzido is not None:
            tokens_traduzidos.append(token_traduzido)
    return tokens_traduzidos


def reconstruir_linha(tokens: list[str]) -> str:
    """Reconstrói uma linha tokenizada evitando espaços incorretos antes de pontuação."""
    partes = []

    for token in tokens:
        if not partes:
            partes.append(token)
            continue

        anterior = partes[-1]
        if token in SEM_ESPACO_ANTES or anterior in SEM_ESPACO_DEPOIS:
            partes.append(token)
        else:
            partes.append(" " + token)

    return "".join(partes)


def traduzir_linha(linha: str, mapa: dict[str, str], marcar_desconhecidos: bool = False) -> str:
    tokens = TOKENIZADOR.tokenize(linha)
    tokens_traduzidos = traduzir_tokens(tokens, mapa, marcar_desconhecidos)
    return reconstruir_linha(tokens_traduzidos)


def traduzir_emojis(texto: str, mapa: dict[str, str] | None = None, marcar_desconhecidos: bool = False) -> str:
    """Tokeniza o texto com NLTK e substitui emojis por descrições textuais em português."""
    mapa = EMOJI_MAP if mapa is None else mapa
    linhas = texto.splitlines()
    linhas_traduzidas = [traduzir_linha(linha, mapa, marcar_desconhecidos) for linha in linhas]
    return "\n".join(linhas_traduzidas).strip()


def ler_conversa(caminho: Path) -> str:
    return caminho.read_text(encoding="utf-8")


def traduzir_arquivo(caminho_entrada: Path, caminho_saida: Path, mapa: dict[str, str] | None = None) -> str:
    texto_original = ler_conversa(caminho_entrada)
    texto_traduzido = traduzir_emojis(texto_original, mapa=mapa)
    caminho_saida.write_text(texto_traduzido + "\n", encoding="utf-8")
    return texto_traduzido

## 4. Execução da tradução

O programa lê a conversa original em UTF-8, traduz os emojis mapeados e grava outro arquivo também em UTF-8.

In [5]:
texto_original = ler_conversa(ARQUIVO_CONVERSA)
texto_traduzido = traduzir_arquivo(ARQUIVO_CONVERSA, ARQUIVO_TRADUZIDO)

print("CONVERSA ORIGINAL:\n")
print(texto_original)
print("\nCONVERSA TRADUZIDA:\n")
print(texto_traduzido)
print(f"\nArquivo traduzido criado: {ARQUIVO_TRADUZIDO}")

CONVERSA ORIGINAL:

[09/05/2026, 19:10] Ana: Oi pessoal! Quem topa estudar PLN hoje? 📚🤓
[09/05/2026, 19:11] Bruno: Eu topo! Mas preciso de café primeiro ☕😂
[09/05/2026, 19:12] Carla: Fechado 👍 Levo os resumos e meu 💻
[09/05/2026, 19:13] Diego: Perfeito! Depois a gente pede 🍕😋
[09/05/2026, 19:14] Ana: Combinado ❤️ Vou mandar o PDF no grupo agora.
[09/05/2026, 19:15] Bruno: Valeu! Se der erro no código eu 😭
[09/05/2026, 19:16] Carla: Calma, a gente resolve 💪✨
[09/05/2026, 19:17] Diego: Bora! 🚀


CONVERSA TRADUZIDA:

[09/05/2026, 19:10] Ana: Oi pessoal! Quem topa estudar PLN hoje? [livros/estudo] [estudioso]
[09/05/2026, 19:11] Bruno: Eu topo! Mas preciso de café primeiro [café] [risada]
[09/05/2026, 19:12] Carla: Fechado [joinha] Levo os resumos e meu [computador]
[09/05/2026, 19:13] Diego: Perfeito! Depois a gente pede [pizza] [saboroso]
[09/05/2026, 19:14] Ana: Combinado [amor] Vou mandar o PDF no grupo agora.
[09/05/2026, 19:15] Bruno: Valeu! Se der erro no código eu [choro]
[09/05/20

## 6. Testes extensivos

Os testes cobrem casos comuns de uso: emoji isolado, emoji colado na pontuação, emojis repetidos, emojis colados entre si, texto sem emoji, coração com seletor de variação solto, emoji com tom de pele, emoji composto com zero-width joiner, emoji desconhecido preservado e opção de marcação de desconhecidos. Também há testes para garantir que a reconstrução da pontuação não introduz espaços errados.

In [6]:
def executar_testes_unitarios() -> None:
    casos = [
        ("Bom dia 😂", "Bom dia [risada]"),
        ("😂😂😂", "[risada] [risada] [risada]"),
        ("Gostei muito!👍", "Gostei muito! [joinha]"),
        ("Gostei muito! 👍🏽", "Gostei muito! [joinha]"),
        ("Amei ❤️", "Amei [amor]"),
        ("Amei ❤", "Amei [amor]"),
        ("Vamos estudar 📚🤓 e tomar ☕.", "Vamos estudar [livros/estudo] [estudioso] e tomar [café]."),
        ("Pizza 🍕😋 depois do trabalho 💻", "Pizza [pizza] [saboroso] depois do trabalho [computador]"),
        ("Sem emojis aqui.", "Sem emojis aqui."),
        ("Emoji desconhecido 🥳", "Emoji desconhecido 🥳"),
        ("Força 💪✨", "Força [força] [brilho]"),
        ("Bora 🚀!", "Bora [empolgação]!"),
        ("Obrigado 🙏👏", "Obrigado [agradecimento] [palmas]"),
        ("[09/05/2026, 19:17] Diego: Bora! 🚀", "[09/05/2026, 19:17] Diego: Bora! [empolgação]"),
    ]

    for entrada, esperado in casos:
        obtido = traduzir_emojis(entrada)
        assert obtido == esperado, f"Falhou para {entrada!r}: esperado {esperado!r}, obtido {obtido!r}"

    assert traduzir_emojis("Surpresa 🥳", marcar_desconhecidos=True) == "Surpresa [emoji desconhecido: 🥳]"
    assert traduzir_emojis("Família 👨‍👩‍👧‍👦", marcar_desconhecidos=True) == "Família [emoji desconhecido: 👨‍👩‍👧‍👦]"
    assert traduzir_emojis("") == ""
    assert traduzir_emojis("😂 texto 😂") == "[risada] texto [risada]"
    assert " !" not in traduzir_emojis("Oi 😂!")
    assert " / " not in traduzir_emojis("[09/05/2026, 19:17] Diego: Bora! 🚀")
    print("Todos os testes unitários passaram.")

executar_testes_unitarios()


Todos os testes unitários passaram.


In [7]:
def contar_ocorrencias_emojis(texto: str, mapa: dict[str, str]) -> Counter:
    contador = Counter()
    for linha in texto.splitlines():
        for token in TOKENIZADOR.tokenize(linha):
            if eh_seletor_variacao_solto(token):
                continue

            if token in mapa:
                contador[token] += 1
                continue

            token_normalizado = normalizar_token_emoji(token)
            if token_normalizado in mapa:
                contador[token_normalizado] += 1
    return contador

contagem_original = contar_ocorrencias_emojis(texto_original, EMOJI_MAP)
print("Emojis encontrados na conversa original:")
for emoji, quantidade in contagem_original.items():
    print(f"{emoji} ({EMOJI_MAP[emoji]}): {quantidade}")

# Garante que todos os emojis conhecidos presentes na conversa foram traduzidos.
for emoji in contagem_original:
    assert emoji not in texto_traduzido, f"O emoji {emoji} ainda aparece no texto traduzido."
    assert f"[{EMOJI_MAP[emoji]}]" in texto_traduzido

# Garante que a estrutura de conversa de WhatsApp foi preservada e sem espaços ruins em data/hora/pontuação.
linhas_originais = texto_original.strip().splitlines()
linhas_traduzidas = texto_traduzido.strip().splitlines()
padrao_whatsapp = re.compile(r"^\[\d{2}/\d{2}/\d{4}, \d{2}:\d{2}\] [^:]+: .+")
assert len(linhas_originais) == len(linhas_traduzidas) == 8
assert all(padrao_whatsapp.match(linha) for linha in linhas_traduzidas)
assert all(" / " not in linha for linha in linhas_traduzidas)
assert all(" !" not in linha and " ?" not in linha and " ," not in linha for linha in linhas_traduzidas)

# Garante que o arquivo de saída também está em UTF-8 e não apresenta sinais comuns de texto corrompido.
texto_saida = ARQUIVO_TRADUZIDO.read_bytes().decode("utf-8")
assert texto_saida.strip() == texto_traduzido
assert "ð" not in texto_saida

print("Testes de integração passaram: arquivo, tradução e formato da conversa estão corretos.")

Emojis encontrados na conversa original:
📚 (livros/estudo): 1
🤓 (estudioso): 1
☕ (café): 1
😂 (risada): 1
👍 (joinha): 1
💻 (computador): 1
🍕 (pizza): 1
😋 (saboroso): 1
❤ (amor): 1
😭 (choro): 1
💪 (força): 1
✨ (brilho): 1
🚀 (empolgação): 1
Testes de integração passaram: arquivo, tradução e formato da conversa estão corretos.
